🏥 SCENARIO: “Healthcare Research Assistant System”
🩺 Background Story
A medical research institute deploys an AI-powered clinical intelligence assistant.
👉 Researchers and doctors can ask:
• 	“What’s the latest research on diabetes treatments?”
• 	“Summarize recent clinical trial results for cancer drugs.”
• 	“Give me a profile of a pharmaceutical company instantly.”
👉 Instead of manually searching journals, trial databases, and company reports,
👉 AI fetches all the data in parallel, analyzes it, and generates a professional research summary.

⚙️ How it works (mapped to your pipeline):
• 	Parallel Data Collection → AI gathers medical news, trial results, and company profiles simultaneously.
• 	LLM Analysis → AI interprets the combined data, highlighting key medical insights.
• 	Report Generation → AI produces a polished, researcher-ready report.

In [ ]:
!pip install groq gradio nest_asyncio

import os
import asyncio
import nest_asyncio
import gradio as gr
from groq import Groq

os.environ["GROQ_API_KEY"] = "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"

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

nest_asyncio.apply()

_original_asyncio_run = asyncio.run

def compatible_asyncio_run(main, *, debug=None, loop_factory=None):
    return _original_asyncio_run(main)

asyncio.run = compatible_asyncio_run


research_updates = {
    "diabetes treatments": [
        "Recent research highlights GLP-1 receptor agonists as a promising treatment pathway for type 2 diabetes.",
        "Studies are exploring personalized diabetes care using AI-driven glucose monitoring systems.",
        "Combination therapies are showing improved glycemic control in selected patient groups."
    ],
    "cancer drugs": [
        "Recent oncology studies emphasize targeted therapies and immunotherapy combinations.",
        "Researchers are investigating better biomarker-based drug selection for cancer patients.",
        "Several new cancer drugs are being evaluated for improved progression-free survival outcomes."
    ],
    "heart disease": [
        "New research focuses on early risk prediction and preventive cardiovascular treatment strategies.",
        "Wearable-based monitoring is being studied for continuous cardiac risk assessment.",
        "Trials continue on novel lipid-lowering and anti-inflammatory therapies."
    ]
}


clinical_trials = {
    "diabetes treatments": [
        {
            "trial_id": "CT-DIA-101",
            "phase": "Phase 3",
            "result": "Improved HbA1c reduction compared to standard therapy",
            "status": "Positive"
        },
        {
            "trial_id": "CT-DIA-202",
            "phase": "Phase 2",
            "result": "Strong patient adherence with digital glucose monitoring support",
            "status": "Encouraging"
        }
    ],
    "cancer drugs": [
        {
            "trial_id": "CT-CAN-301",
            "phase": "Phase 3",
            "result": "Extended progression-free survival in selected patient groups",
            "status": "Positive"
        },
        {
            "trial_id": "CT-CAN-404",
            "phase": "Phase 2",
            "result": "Combination therapy showed manageable safety profile",
            "status": "Promising"
        }
    ],
    "heart disease": [
        {
            "trial_id": "CT-CARD-111",
            "phase": "Phase 3",
            "result": "Reduced cardiovascular risk markers in high-risk adults",
            "status": "Positive"
        },
        {
            "trial_id": "CT-CARD-222",
            "phase": "Phase 2",
            "result": "Improved adherence through wearable-linked interventions",
            "status": "Promising"
        }
    ]
}


pharma_profiles = {
    "Pfizer": {
        "industry": "Biopharmaceuticals",
        "hq": "New York, USA",
        "employees": "Approx. 88000",
        "summary": "Pfizer develops and manufactures medicines and vaccines across oncology, internal medicine, and rare diseases."
    },
    "Novartis": {
        "industry": "Pharmaceuticals",
        "hq": "Basel, Switzerland",
        "employees": "Approx. 76000",
        "summary": "Novartis focuses on innovative medicines across cardiovascular, immunology, neuroscience, and oncology areas."
    },
    "Roche": {
        "industry": "Pharmaceuticals and Diagnostics",
        "hq": "Basel, Switzerland",
        "employees": "Approx. 100000",
        "summary": "Roche operates across diagnostics and pharmaceuticals with strong research presence in oncology and precision medicine."
    }
}


async def fetch_medical_research(topic):
    await asyncio.sleep(1)

    if topic in research_updates:
        text = f"Latest Medical Research on {topic.title()}:\n"
        for i, item in enumerate(research_updates[topic], start=1):
            text += f"- {i}. {item}\n"
        return text.strip()

    return f"No medical research updates available for {topic}."


async def fetch_clinical_trials(topic):
    await asyncio.sleep(1)

    if topic in clinical_trials:
        text = f"Recent Clinical Trial Results for {topic.title()}:\n"
        for trial in clinical_trials[topic]:
            text += (
                f"- Trial ID: {trial['trial_id']}, "
                f"Phase: {trial['phase']}, "
                f"Result: {trial['result']}, "
                f"Status: {trial['status']}\n"
            )
        return text.strip()

    return f"No clinical trial data available for {topic}."


async def fetch_pharma_profile(company):
    await asyncio.sleep(1)

    if company in pharma_profiles:
        profile = pharma_profiles[company]
        return (
            f"Pharmaceutical Company Profile:\n"
            f"- Company: {company}\n"
            f"- Industry: {profile['industry']}\n"
            f"- Headquarters: {profile['hq']}\n"
            f"- Employees: {profile['employees']}\n"
            f"- Summary: {profile['summary']}"
        )

    return f"No company profile available for {company}."


async def parallel_research(topic, company):
    results = await asyncio.gather(
        fetch_medical_research(topic),
        fetch_clinical_trials(topic),
        fetch_pharma_profile(company),
        return_exceptions=True
    )

    research, trials, profile = results

    return {
        "research": research if not isinstance(research, Exception) else "Research data unavailable",
        "trials": trials if not isinstance(trials, Exception) else "Clinical trial data unavailable",
        "profile": profile if not isinstance(profile, Exception) else "Company profile unavailable"
    }


def decide_intent(user_query):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{
            "role": "user",
            "content": f"""
You are a healthcare research assistant intent classifier.

Classify the user's query into exactly one of these:
- research
- trials
- profile
- full_report

Rules:
- "latest research" -> research
- "clinical trial" or "trial results" -> trials
- "company profile" or "pharmaceutical company profile" -> profile
- "full report", "research summary", "complete report" -> full_report

Return exactly one label.

User Query: {user_query}
"""
        }]
    )
    return response.choices[0].message.content.strip().lower()


def extract_topic(user_query):
    possible_topics = list(research_updates.keys())
    for topic in possible_topics:
        if topic.lower() in user_query.lower():
            return topic
    return None


def extract_company(user_query):
    possible_companies = list(pharma_profiles.keys())
    for company in possible_companies:
        if company.lower() in user_query.lower():
            return company
    return None


def analyse_healthcare_data(text, topic, company):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{
            "role": "user",
            "content": f"""
Analyze the following healthcare research intelligence for topic '{topic}' and company '{company}'.

Provide:
1. Key clinical/research summary
2. Important scientific observations
3. Trial-related insight
4. Risks, limitations, or opportunities
5. Simple researcher-friendly explanation

Data:
{text}
"""
        }]
    )
    return response.choices[0].message.content


def generate_healthcare_report(analysis, topic, company):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{
            "role": "user",
            "content": f"""
Create a professional healthcare research summary report.

Topic: {topic}
Company: {company}

Use this analysis:
{analysis}

Keep it:
- clear
- structured
- concise but informative
- suitable for researchers and doctors
"""
        }]
    )
    return response.choices[0].message.content


async def full_pipeline(topic, company, user_query):
    if not topic and not company:
        return "Please mention a supported research topic or pharmaceutical company."

    if not topic:
        topic = extract_topic(user_query)

    if not company:
        company = extract_company(user_query)

    if not topic:
        topic = "diabetes treatments"

    if not company:
        company = "Pfizer"

    data = await parallel_research(topic, company)
    intent = decide_intent(user_query)

    if intent == "research":
        combined_text = data["research"]
    elif intent == "trials":
        combined_text = data["trials"]
    elif intent == "profile":
        combined_text = data["profile"]
    else:
        combined_text = f"""
{data['research']}

{data['trials']}

{data['profile']}
""".strip()

    analysis = analyse_healthcare_data(combined_text, topic, company)
    report = generate_healthcare_report(analysis, topic, company)

    final_output = f"""
==============================
HEALTHCARE RESEARCH ASSISTANT OUTPUT
==============================

Topic: {topic}
Company: {company}
Detected Intent: {intent}

RAW DATA:
{combined_text}

--------------------------------
AI ANALYSIS + RESEARCH SUMMARY:
--------------------------------
{report}
"""
    return final_output.strip()


def run_normal_mode():
    print("Healthcare Research Assistant System")
    topic = input("Enter Research Topic (diabetes treatments / cancer drugs / heart disease): ").strip()
    company = input("Enter Pharma Company (Pfizer / Novartis / Roche): ").strip()
    user_question = input("Ask your question: ").strip()

    if not topic:
        topic = extract_topic(user_question)

    if not company:
        company = extract_company(user_question)

    if not user_question:
        print("Please enter a question.")
        return

    result = asyncio.run(full_pipeline(topic, company, user_question))
    print("\nFINAL OUTPUT:\n")
    print(result)


def healthcare_assistant_ui(topic, company, user_query):
    topic = topic.strip()
    company = company.strip()
    user_query = user_query.strip()

    if not topic:
        topic = extract_topic(user_query)

    if not company:
        company = extract_company(user_query)

    if not user_query:
        return "Please enter a question."

    return asyncio.run(full_pipeline(topic, company, user_query))


run_normal_mode()


with gr.Blocks() as demo:
    gr.Markdown("# Healthcare Research Assistant System")
    gr.Markdown("""
Supported research topics:
- diabetes treatments
- cancer drugs
- heart disease

Supported pharma companies:
- Pfizer
- Novartis
- Roche

Example questions:
- What's the latest research on diabetes treatments?
- Summarize recent clinical trial results for cancer drugs.
- Give me a profile of Pfizer instantly.
- Give me a full healthcare research report on cancer drugs and Roche.
""")

    topic_input = gr.Textbox(
        label="Enter Research Topic",
        placeholder="Example: diabetes treatments"
    )

    company_input = gr.Textbox(
        label="Enter Pharma Company",
        placeholder="Example: Pfizer"
    )

    query_input = gr.Textbox(
        label="Ask your question",
        placeholder="Example: Summarize recent clinical trial results for cancer drugs."
    )

    output_box = gr.Textbox(
        label="Assistant Response",
        lines=24
    )

    submit_btn = gr.Button("Get Research Summary")

    submit_btn.click(
        fn=healthcare_assistant_ui,
        inputs=[topic_input, company_input, query_input],
        outputs=output_box
    )

demo.launch(share=True, debug=True)

Healthcare Research Assistant System

FINAL OUTPUT:

HEALTHCARE RESEARCH ASSISTANT OUTPUT

Topic: diabetes treatments Pfizer What's the latest research on diabetes treatments?
Company: Roche
Detected Intent: profile

RAW DATA:
Pharmaceutical Company Profile:
- Company: Roche
- Industry: Pharmaceuticals and Diagnostics
- Headquarters: Basel, Switzerland
- Employees: Approx. 100000
- Summary: Roche operates across diagnostics and pharmaceuticals with strong research presence in oncology and precision medicine.

--------------------------------
AI ANALYSIS + RESEARCH SUMMARY:
--------------------------------
**Diabetes Treatments Research Summary Report: Roche**

**Introduction:**
As a leading pharmaceutical company, Roche has a significant presence in the healthcare industry. While their primary research focus is on oncology and precision medicine, this report aims to summarize the latest research on diabetes treatments and explore potential opportunities for Roche in this therapeutic ar